# LLM Call — Unit Test

Smoke test for whatever provider `.env` currently points at. Run it after
changing provider, key or model, before blaming a node for a failure that is
really a bad credential.

Three checks, because the pipeline needs all three: settings are readable, a
plain chat call works, and a structured-output call works — every node binds a
schema through function calling, which some models refuse while reasoning is on.

**Pre-requisites:** `RULESBANK_PROVIDER` in `.env` plus the matching key
(`OPENROUTER_API_KEY` or `OPENAI_API_KEY`). No Qdrant, no fixtures.

In [ ]:
# Step 1 — Imports and the settings actually in effect
#
# Printing these first turns the usual "which key did it pick up?" question
# into something the notebook answers before any request is made.

from pydantic import BaseModel, Field

from openapi_rulesbank.config import get_logger
from openapi_rulesbank.config.settings import (
    LLM_MAX_RETRIES,
    LLM_REQUEST_TIMEOUT,
    MODEL,
    OPENROUTER_BASE_URL,
    PROVIDER,
    REASONING_EFFORT,
    TEMPERATURE,
)

logger = get_logger(__name__)

endpoint = OPENROUTER_BASE_URL if PROVIDER == "openrouter" else "OpenAI default"
logger.info(f"provider    : {PROVIDER}")
logger.info(f"model       : {MODEL}")
logger.info(f"endpoint    : {endpoint}")
logger.info(f"temperature : {TEMPERATURE}")
logger.info(f"reasoning   : {REASONING_EFFORT or '(unset)'}")
logger.info(f"timeout     : {LLM_REQUEST_TIMEOUT}s, retries: {LLM_MAX_RETRIES}")

In [ ]:
# Step 2 — Import the shared client. The nodes import this same module-level
# instance, so a missing key or an unknown provider raises right here — with the
# message naming the variable to fix — before any request leaves the machine.

from openapi_rulesbank.config.llm_config import llm

logger.info(f"client ready : {type(llm).__name__}")
logger.info(f"base_url     : {llm.openai_api_base or 'OpenAI default'}")
logger.info(f"extra_body   : {getattr(llm, 'extra_body', None)}")
logger.info(f"headers      : {getattr(llm, 'default_headers', None)}")

In [ ]:
# Step 3 — Plain chat call. Proves the key, the base URL and the model id are
# all accepted; a wrong vendor prefix (openai/gpt-... on OpenAI, or a bare name
# on OpenRouter) fails here.

reply = llm.invoke("How many r's are in the word 'strawberry'? Answer in one line.")
logger.info(f"reply : {reply.content}")

usage = getattr(reply, "usage_metadata", None)
if usage:
    logger.info(f"tokens: {usage}")

In [ ]:
# Step 4 — Structured output. The nodes never call the model bare: they bind a
# Pydantic schema and read back fields. Reasoning models reject function calling
# on /v1/chat/completions unless reasoning is off, so this cell is the one that
# catches a RULESBANK_REASONING_EFFORT the pipeline cannot live with.


class Answer(BaseModel):
    """Minimal schema — the point is the call shape, not the content."""

    word: str = Field(description="The word that was counted")
    count: int = Field(description="How many times the letter appears")


structured = llm.with_structured_output(Answer).invoke(
    "How many r's are in the word 'strawberry'?"
)
logger.info(f"structured: {structured!r}")

assert isinstance(structured, Answer), "structured output did not come back typed"
logger.info(f"OK — {PROVIDER}/{MODEL} answers both plain and structured calls.")